In [172]:
import fastf1
from fastf1.core import Laps

import plotly.graph_objects as go
import plotly.express as px
from plotly.offline import init_notebook_mode

import numpy as np
import pandas as pd

init_notebook_mode(connected=True)

## Notebook Configurations

In [223]:
# Columns that are dropped from the Raw Laps Dataframe
DROP_COLS = [
    "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime", 
    "LapStartTime", "LapStartDate", "FastF1Generated", "IsAccurate", "DeletedReason"
]

# Order for the Stint Category to be followed in the Facets
STINT_ORDER = {
    "Stint": [1.0, 2.0, 3.0, 4.0]
}

# Hover Data to be displayed for each datapoint for each driver
HOVER_DATA = ["Driver", "LapNumber", "Position", "Compound", "TrackStatus"]

## Notebook Utilities
- Especially useful for building cohesive classes for easier plotting and construction of data pipelines.

In [ ]:
class DataUtils:
    """This class provides the semantic structure for multiple data utility functions to
    ensure the consistency of Data Cleaning and Engineering by constructing a robust, customisable
    data pipeline."""

    # ======================== Class Level variables to perform Tyre Degradation Ops ========================
    # Maximum Fuel Load permissible for the car (110 KG)
    max_fuel_load = 110

    # Fuel Burn kg - per/second (27.8g per/sec)
    fuel_burn_rate = 27.8 * 1e-3

    # Fuel Load to Weight Constant (10Kg => 0.3sec)
    fuel_load_constant = 0.3 * 1e-1

    # Internal Combustion Contribution through a Lap
    ice_const = 0.8

    # ======================== Semantically Relevant Methods for operating on Data ========================

    @staticmethod
    def get_processed_laps(race_laps: Laps, drop_cols: list = DROP_COLS) -> Laps:
        """Processes the raw lap data and provides a more clean version of the race laps
        while maintaining the FastF1 Laps structure."""

        # Dropping InPlace
        race_laps = race_laps.drop(drop_cols, axis=1)

        # Changing the Lap and Sector Times from TimeDelta to Seconds
        lap_cols = ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]
        for col in lap_cols:
            race_laps[col] = race_laps[col].apply(lambda x: x.total_seconds())

        return race_laps
    
    @staticmethod
    def get_valid_quick_laps(race_laps: Laps) -> Laps:
        """Retrieves all the laps from the race session which are not in / out - laps and 
        are laps set at race-speed by default a threshold of 1.07% the best laptime."""
        
        # Until I have a way to model the fuel flow, stick to WO-Box for filtering laps
        return (
            race_laps
            .pick_wo_box()
            .pick_quicklaps(threshold=1.1).copy()
        )
    
    @staticmethod
    def add_tyre_deg_variables(race_laps: Laps) -> Laps:
        """Utilised the processed raw lap data to calculate necessary tyre degradation variables:
        - Fuel Burn Per/Lap
        - Cumulative Fuel Burn for the race
        - Fuel-Corrected Lap Time for isolated car & driver level basis for tyre degradation 
        independant of the weather variables."""

        # Accessing valid quick race laps
        quick_race_laps = DataUtils.get_valid_quick_laps(race_laps=race_laps)

        # Fuel burnt during any given lap
        quick_race_laps["LapFuelBurnt"] = quick_race_laps["LapTime"] * DataUtils.fuel_burn_rate * DataUtils.ice_const

        # Cumulative fuel burnt by a driver during the race
        quick_race_laps["CumulativeFuelBurnt"] = quick_race_laps.groupby(["Driver"])["LapFuelBurnt"].cumsum()

        # Fuel burnt to the nth lap
        fuel_burnt_before_lap_n = quick_race_laps.groupby(["Driver"])["CumulativeFuelBurnt"].shift(1).fillna(0)

        # The Fuel Aware Laptime
        fuel_load_start_lap_n = DataUtils.max_fuel_load - fuel_burnt_before_lap_n
        fuel_penality_for_lap = fuel_load_start_lap_n * DataUtils.fuel_load_constant
        quick_race_laps["FuelAwareLapTime"] = quick_race_laps["LapTime"] - fuel_penality_for_lap

        return quick_race_laps

In [225]:
class PlotUtils:
    """This class provides the semantic structure for multiple plotting functions to ease the 
    visualisations that are made throughout the notebook."""

    @staticmethod
    def plot_line_chart(
        data_frame: pd.DataFrame, 
        X: np.ndarray | pd.Series | str, 
        y: np.ndarray | pd.Series | str, 
        label: np.ndarray | pd.Series | str | None,
        title: str,
        **kwargs
    ) -> None:
        """A helper function to efficiently plot multiple line charts using Plotly through out the notebook."""

        # Plotly Line Chart
        fig = px.line(
            data_frame=data_frame,
            x=X,
            y=y,
            color=label,
            labels=label,
            title=title,
            width=1540,
            height=kwargs["height"] if kwargs else 720,
            facet_row=kwargs["facet_row"] if kwargs else None,
            category_orders=kwargs["category_orders"] if kwargs else None,
            hover_data=kwargs["hover_data"] if kwargs else None
        )

        fig.show()

In [226]:
# Creating instances of all the Utility Objects
data_pipe = DataUtils()
plotter = PlotUtils()

## Loading the Data

In [155]:
brazil_2025 = fastf1.get_event(
    year=2025,
    gp="Brazil"
)

brazil_2025

RoundNumber                                                         21
Country                                                         Brazil
Location                                                     São Paulo
OfficialEventName    FORMULA 1 MSC CRUISES GRANDE PRÊMIO DE SÃO PAU...
EventDate                                          2025-11-09 00:00:00
EventName                                         São Paulo Grand Prix
EventFormat                                          sprint_qualifying
Session1                                                    Practice 1
Session1Date                                 2025-11-07 11:30:00-03:00
Session1DateUtc                                    2025-11-07 14:30:00
Session2                                             Sprint Qualifying
Session2Date                                 2025-11-07 15:30:00-03:00
Session2DateUtc                                    2025-11-07 18:30:00
Session3                                                        Sprint
Sessio

## Accessing the Race Session

In [156]:
brazil_race = brazil_2025.get_race()

brazil_race.load(
    laps=True,
    telemetry=True,
    weather=True,
    messages=True
)

core           INFO 	Loading data for São Paulo Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 4 completed the race distance 00:00.010000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['4', '12', '1', '63', '81', '87', '30', '6', '27', '10', '23', '3

In [227]:
# Accessing the race laps
race_laps = brazil_race.laps

# Cleaning the Data using the Pipeline
race_laps = data_pipe.get_processed_laps(race_laps=race_laps)
race_laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,TrackStatus,Position,Deleted
0,0 days 00:58:21.863000,VER,1,87.191,1.0,1.0,0 days 00:57:08.347000,NaT,NaN,39.288,...,305.0,NaN,False,HARD,1.0,True,Red Bull Racing,12,18.0,False
1,0 days 01:00:22.276000,VER,1,120.413,2.0,1.0,NaT,NaT,21.320,75.859,...,243.0,271.0,True,HARD,2.0,True,Red Bull Racing,24,17.0,False
2,0 days 01:02:11.283000,VER,1,109.007,3.0,1.0,NaT,NaT,21.675,58.682,...,253.0,258.0,True,HARD,3.0,True,Red Bull Racing,4,16.0,False
3,0 days 01:03:57.903000,VER,1,106.620,4.0,1.0,NaT,NaT,25.318,59.748,...,233.0,237.0,True,HARD,4.0,True,Red Bull Racing,4,16.0,False
4,0 days 01:05:45.792000,VER,1,107.889,5.0,1.0,NaT,NaT,30.078,50.463,...,291.0,247.0,False,HARD,5.0,True,Red Bull Racing,41,16.0,False


In [228]:
# Querying the data for Top 5 Drivers
top_5_drivers = race_laps[
    (race_laps["Position"] >= 1) &
    (race_laps["Position"] <= 5) &
    (race_laps["LapNumber"] == 70)
]["Driver"].unique()

# Plotting the Position Progression of Drivers in Top 5
plotter.plot_line_chart(
    data_frame=race_laps[
        race_laps["Driver"].isin(top_5_drivers)
    ],
    X="LapNumber",
    y="Position",
    label="Driver",
    title="Driver Position Progression throughout the Race (Top 5)"
)

## Analysing the Tyre Degradation

**Key Assumptions Made**
- Due to the lack of hybrid energy and ICE deployment data I have ignored the contribution of the hybrid in pace calculation and enforced a weighted constant for ICE contribution as per the previous regulations (2022 - 2025) where:
    - *ICE contributes 80% of the power while the hybrid system contributes 20% of the power*
- While car telemetry data is available, it wasn't available for the race session and hence the fuel burn calculation is laid out for the worst case scenario i.e, the max fuel-flow is being utilsed every second of the lap. Thus, every lap in this racing world is hence is near push lap which punishes bad tyre degradation further.
- I have chosen to focus on the top 5 by calculating the mean position of a driver throughout the race (lower is better) since the front of the pack is in resonably clean air and in the contention for a race win leaving nothing in the cockpit.

In [229]:
# Applying the transforms for creating tyre degradation features
quick_race_laps = data_pipe.add_tyre_deg_variables(race_laps=race_laps)

# Selecting the top 5 finishers
filtered_quick_race_laps = quick_race_laps.pick_drivers(top_5_drivers)
filtered_quick_race_laps

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,Compound,TyreLife,FreshTyre,Team,TrackStatus,Position,Deleted,LapFuelBurnt,CumulativeFuelBurnt,FuelAwareLapTime
1,0 days 01:00:22.276000,VER,1,120.413,2.0,1.0,NaT,NaT,21.320,75.859,...,HARD,2.0,True,Red Bull Racing,24,17.0,False,2.677985,2.677985,117.113000
2,0 days 01:02:11.283000,VER,1,109.007,3.0,1.0,NaT,NaT,21.675,58.682,...,HARD,3.0,True,Red Bull Racing,4,16.0,False,2.424316,5.102301,105.787340
3,0 days 01:03:57.903000,VER,1,106.620,4.0,1.0,NaT,NaT,25.318,59.748,...,HARD,4.0,True,Red Bull Racing,4,16.0,False,2.371229,7.473530,103.473069
4,0 days 01:05:45.792000,VER,1,107.889,5.0,1.0,NaT,NaT,30.078,50.463,...,HARD,5.0,True,Red Bull Racing,41,16.0,False,2.399451,9.872981,104.813206
5,0 days 01:07:04.287000,VER,1,78.495,6.0,1.0,NaT,NaT,20.374,40.996,...,HARD,6.0,True,Red Bull Racing,12,13.0,False,1.745729,11.618710,75.491189
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,0 days 02:24:17.396000,PIA,81,73.734,67.0,3.0,NaT,NaT,18.857,38.111,...,MEDIUM,25.0,False,McLaren,1,5.0,False,1.639844,108.745327,73.647164
1176,0 days 02:25:31.057000,PIA,81,73.661,68.0,3.0,NaT,NaT,18.934,38.083,...,MEDIUM,26.0,False,McLaren,1,5.0,False,1.638221,110.383547,73.623360
1177,0 days 02:26:44.399000,PIA,81,73.342,69.0,3.0,NaT,NaT,18.663,38.098,...,MEDIUM,27.0,False,McLaren,1,5.0,False,1.631126,112.014673,73.353506
1178,0 days 02:27:58.101000,PIA,81,73.702,70.0,3.0,NaT,NaT,18.803,38.093,...,MEDIUM,28.0,False,McLaren,1,5.0,False,1.639132,113.653806,73.762440


In [230]:
# Verifying the fuel consumption for each of the drivers
filtered_quick_race_laps.groupby(["Driver", "Stint"])["CumulativeFuelBurnt"].last()

Driver  Stint
ANT     1.0       37.956808
        2.0       77.723952
        3.0      115.378095
NOR     1.0       52.772228
        2.0       82.525434
        3.0      115.195549
PIA     1.0       66.300665
        2.0       84.311618
        3.0      115.297698
RUS     1.0       59.793352
        2.0       79.512915
        3.0      115.588508
VER     1.0       11.618710
        2.0       53.283926
        3.0       82.802678
        4.0      108.861108
Name: CumulativeFuelBurnt, dtype: float64

## Visualisations

### Full Lap Pace with respect to stint for the Front Runners

In [231]:
plotter.plot_line_chart(
    data_frame=filtered_quick_race_laps,
    X="LapNumber",
    y="FuelAwareLapTime",
    label="Driver",
    title="Pure Pace of the Front Runner over a Full Lap in Race Distance",
    facet_row="Stint",
    category_orders=STINT_ORDER,
    hover_data=HOVER_DATA,
    height=1080,
)

### Understanding Tyre Degradation with respect to Track Layout by Sectors

>Source: F1

![Image of the Interlagos Circuit Layout](../../assets/Brazil_Circuit.png)

### Sector-Wise Pace for each of the Front Runners with respect Stint

#### Sector - 1

In [204]:
plotter.plot_line_chart(
    data_frame=filtered_quick_race_laps,
    X="LapNumber",
    y="Sector1Time",
    label="Driver",
    title="Pace of the Front Runners in Sector - 1",
    facet_row="Stint",
    category_orders=STINT_ORDER,
    hover_data=HOVER_DATA,
    height=1080,
)

#### Sector - 2

In [206]:
plotter.plot_line_chart(
    data_frame=filtered_quick_race_laps,
    X="LapNumber",
    y="Sector2Time",
    label="Driver",
    title="Pace of the Front Runners in Sector - 2",
    facet_row="Stint",
    category_orders=STINT_ORDER,
    hover_data=HOVER_DATA,
    height=1080,
)

#### Sector - 3

In [205]:
plotter.plot_line_chart(
    data_frame=filtered_quick_race_laps,
    X="LapNumber",
    y="Sector3Time",
    label="Driver",
    title="Pace of the Front Runners in Sector - 3",
    facet_row="Stint",
    category_orders=STINT_ORDER,
    hover_data=HOVER_DATA,
    height=1080,
)